# 06 - Tolerances: how precisely must we build it?

**Goal:** find which build errors matter, how much they hurt, and the tolerance each needs.

The controller knows the *drawing*; the real table is slightly off (magnet strength spread,
misplaced coils, tile misalignment, tilt, sensor offset...). We build many random "as-built"
tables and measure what each error costs (see `docs/tolerances.md`):

| Impact metric | Meaning |
|---|---|
| `force_error_pct` | unexpected vertical force, % of weight |
| `lateral_accel_g` | unexpected sideways push, in g |
| `torque_error_pct` | unexpected tilting torque |
| `current_increase_pct` | extra current needed once calibrated |
| `lift_margin_loss_pct` | lost lift capacity |

Assumed tolerances and the acceptable thresholds live in the config under `tolerances:` -
**agree the thresholds as a team**; they are starting points.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from maglev_sim import *
from maglev_sim import plotting

In [ ]:
from maglev_sim.tolerance import tolerance_report, run_trials, as_built, TOLERANCES
s = build_system(load_config("../configs/baseline.yaml"))
tr = tolerance_report(s, trials=8)   # ~1-2 minutes; raise trials for final numbers
print(tr.text())

In [ ]:
plotting.plot_tolerance(tr);

**Reading it:** left plot ranks the tolerances (red bars break a threshold on their own at the
assumed value). The table gives `max_allowed` for each - that is the number to put on the
drawing / assembly instructions. `limited_by` says which effect sets it.

## Does it still levitate? Simulate a random as-built table
The controller still uses the nominal design; the 'plant' is a random build.

In [ ]:
rng = np.random.default_rng(1)
values = {k: v for k, v in s.cfg["tolerances"]["values"].items()}
plant, pose_err = as_built(s, values, rng)
p0 = np.r_[s.grid.tile_center(0), s.gap]
p1 = np.r_[s.grid.tile_center(1), s.gap]
nominal = simulate(s, Trajectory.move(p0, p1, 0.1, 0.8), duration=1.1)
built = simulate(s, Trajectory.move(p0, p1, 0.1, 0.8), duration=1.1, plant=plant,
                 plant_pose_error=np.r_[0, 0, pose_err[2], 0, 0, 0], sensor_bias=np.r_[pose_err[:2], 0])
print("nominal :", nominal.summary())
print("as-built:", built.summary())

## Tighten or loosen one tolerance
E.g. what if tiles are aligned with dowel pins to +/-0.1 mm instead of +/-0.5 mm?

In [ ]:
for tile_tol in [0.1, 0.25, 0.5, 1.0]:
    df = run_trials(s, {"tile_offset_mm": tile_tol}, trials=8)
    print(f"tile offset +/-{tile_tol} mm -> p95 force error {np.percentile(df.force_error_pct, 95):.1f} %, "
          f"lateral {np.percentile(df.lateral_accel_g, 95):.3f} g, extra current {np.percentile(df.current_increase_pct, 95):.1f} %")